<a href="https://colab.research.google.com/github/gyanellaagreda-bd/MINERIA-DE-DATOS---2026-1/blob/main/Copia_de_LAB_S04_MDD_PSAYAN_2026_01.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🔬 LABORATORIO N° 04 — Minería de Datos
## Preparación de Datos (Fase 3 de CRISP-DM)

| | |
|---|---|
| **Curso** | Minería de Datos — C64893 |
| **Semana** | 4 |
| **Docente** | Pilar Rocío Sayán Mejía |
| **Institución** | TECSUP — Pasión por la Tecnología |
| **Semestre** | 2026-I |

---
### 📋 Instrucciones generales
- Ejecuta **todas** las celdas de código en orden.
- Responde las preguntas en las celdas de texto (Markdown) asignadas.
- Al finalizar, descarga el notebook (.ipynb) con todas las celdas ejecutadas.

### 📚 Referencias
- Gironés Roig, J. et al. (2017). *Minería de datos: modelos y algoritmos*. Editorial UOC. Cap. 3.
- Géron, A. (2022). *Hands-On Machine Learning* (3.ª ed.). O'Reilly Media. Cap. 2.
- Ilyas, I. F. y Chu, X. (2019). *Cleaning Data for Data Science*. ACM.
- Müller, A. C. y Guido, S. (2016). *Introduction to Machine Learning with Python*. O'Reilly Media. Cap. 3.

---
# ACTIVIDAD 1: Revisión de Conceptos — Preparación de Datos

| N° | Concepto | Definición |
|----|----------|------------|
| 1 | ¿Por qué es importante la preparación de datos en CRISP-DM? | Garantiza que los datos sean consistentes, completos y útiles antes del modelado. Datos de baja calidad producen modelos defectuosos sin importar el algoritmo (principio GIGO). Es la fase que más tiempo consume y mayor impacto tiene en el rendimiento final. |
| 2 | Principio GIGO (Garbage In, Garbage Out) | Si los datos de entrada son incorrectos, incompletos o sesgados, el modelo producirá resultados igualmente defectuosos. La calidad del modelo depende directamente de la calidad de los datos. |
| 3 | Imputación de valores nulos | Técnica para reemplazar valores faltantes con estimaciones: media o mediana (variables numéricas), moda (categóricas), o métodos avanzados como KNN o regresión. Evita perder registros completos por datos faltantes parciales. |
| 4 | Detección de outliers con IQR | Valores fuera del rango [Q1 − 1.5·IQR, Q3 + 1.5·IQR] se consideran atípicos. IQR = Q3 − Q1. Es un método robusto que no asume distribución normal. |
| 5 | Normalización Min-Max vs Estandarización Z-score | Min-Max escala valores al rango [0, 1]; sensible a outliers. Z-score transforma a media=0 y desviación estándar=1; más robusto ante valores extremos. Se usan según el algoritmo y la distribución de los datos. |
| 6 | One-Hot Encoding | Convierte variables categóricas en columnas binarias (0/1), una por cada categoría. Permite que los algoritmos procesen variables no numéricas. Se usa drop_first=True para evitar multicolinealidad. |
| 7 | Feature Engineering | Creación de nuevas variables a partir de las existentes para capturar relaciones ocultas y mejorar el poder predictivo del modelo. Requiere conocimiento del negocio y del dominio del problema. |
| 8 | Discretización | Transformar una variable continua en intervalos o rangos discretos (ej. tenure → Nuevo/Corto/Medio/Largo). Útil para capturar relaciones no lineales y mejorar la interpretabilidad. |
| 9 | Selección de atributos (métodos de filtro) | Técnicas que evalúan la relevancia de cada variable respecto a la variable objetivo sin entrenar el modelo (ej. información mutua, chi cuadrado, correlación). Reduce ruido y dimensionalidad. |
| 10 | PCA (Análisis de Componentes Principales) | Técnica de reducción de dimensionalidad que proyecta los datos en nuevos ejes (componentes) que maximizan la varianza explicada. Elimina redundancia entre variables correlacionadas. |
| 11 | ¿Qué es Data Leakage? | Ocurre cuando información del conjunto de prueba o del futuro contamina el entrenamiento, generando métricas infladas irreales. Se evita dividiendo train/test antes de cualquier transformación y ajustando los transformadores solo con datos de entrenamiento. |
| 12 | Pipeline de preparación en scikit-learn | Objeto que encadena transformaciones y modelo en un flujo reproducible y ordenado. Garantiza que cada paso se aplique correctamente en entrenamiento y predicción, evitando data leakage y errores manuales. |

---
# ACTIVIDAD 2: Pipeline de Preparación de Datos
> *Ref: Gironés Roig, J. et al. (2017). Minería de datos. Editorial UOC. Cap. 3.*

---
## 🔹 Paso 1: Carga del dataset y detección de problemas
*Ref: Ilyas, I. F. y Chu, X. (2019). Cleaning Data for Data Science. ACM.*

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Cargar dataset
url = 'https://raw.githubusercontent.com/IBM/telco-customer-churn-on-icp4d/master/data/Telco-Customer-Churn.csv'
df = pd.read_csv(url)

# Detección de valores nulos
print('VALORES NULOS POR COLUMNA:')
print(df.isnull().sum())

# Detección de duplicados
print(f'\nDUPLICADOS: {df.duplicated().sum()} filas')

# Detectar problemas en TotalCharges
df['TotalCharges'] = pd.to_numeric(df['TotalCharges'], errors='coerce')
nulos_totalcharges = df['TotalCharges'].isnull().sum()
print(f'\nVALORES NULOS EN TotalCharges DESPUÉS DE CONVERSIÓN: {nulos_totalcharges}')

VALORES NULOS POR COLUMNA:
customerID          0
gender              0
SeniorCitizen       0
Partner             0
Dependents          0
tenure              0
PhoneService        0
MultipleLines       0
InternetService     0
OnlineSecurity      0
OnlineBackup        0
DeviceProtection    0
TechSupport         0
StreamingTV         0
StreamingMovies     0
Contract            0
PaperlessBilling    0
PaymentMethod       0
MonthlyCharges      0
TotalCharges        0
Churn               0
dtype: int64

DUPLICADOS: 0 filas

VALORES NULOS EN TotalCharges DESPUÉS DE CONVERSIÓN: 11


### ✏️ Pregunta 1
**¿Cuántos valores nulos tiene la variable TotalCharges después de la conversión? ¿Qué método de imputación usarías y por qué?**

> **Respuesta:** *Hay 11 valores nulos. Se usa imputación por mediana porque TotalCharges es una distribución sesgada (clientes con tenure=0 tienen cargo total=0), y la mediana no se ve distorsionada por outliers. La media sobreestimaría el valor real de esos registros.*

---
## 🔹 Paso 2: Detección y tratamiento de outliers
*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

In [ ]:
def detectar_outliers_iqr(df, columna):
    Q1 = df[columna].quantile(0.25)
    Q3 = df[columna].quantile(0.75)
    IQR = Q3 - Q1
    limite_inferior = Q1 - 1.5 * IQR
    limite_superior = Q3 + 1.5 * IQR
    outliers = df[(df[columna] < limite_inferior) | (df[columna] > limite_superior)]
    return len(outliers), limite_inferior, limite_superior

vars_numericas = ['tenure', 'MonthlyCharges', 'TotalCharges']
for var in vars_numericas:
    n_outliers, li, ls = detectar_outliers_iqr(df, var)
    print(f'{var}: {n_outliers} outliers (límites: {li:.2f} - {ls:.2f})')

tenure: 0 outliers (límites: -60.00 - 124.00)
MonthlyCharges: 0 outliers (límites: -46.02 - 171.38)
TotalCharges: 0 outliers (límites: -4688.48 - 8884.67)


### ✏️ Pregunta 2
**¿Existen outliers en las variables numéricas? Si hay outliers, ¿deberían eliminarse o transformarse? Justifica.**

> **Respuesta:** *No existen outliers en ninguna de las tres variables numéricas (tenure, MonthlyCharges, TotalCharges) según el método IQR. Los límites son amplios porque la distribución de los datos es dispersa pero uniforme. No se requiere ninguna acción de tratamiento.*

---
## 🔹 Paso 3: Normalización y estandarización
*Ref: Müller, A. C. y Guido, S. (2016). Introduction to Machine Learning with Python. Cap. 3.*

In [ ]:
from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Imputar nulos primero
df['TotalCharges'] = df['TotalCharges'].fillna(df['TotalCharges'].median())

# Normalización Min-Max (0-1)
minmax = MinMaxScaler()
df_minmax = pd.DataFrame(minmax.fit_transform(df[vars_numericas]), columns=vars_numericas)

# Estandarización Z-score (media=0, std=1)
zscore = StandardScaler()
df_zscore = pd.DataFrame(zscore.fit_transform(df[vars_numericas]), columns=vars_numericas)

print('ORIGINAL - Estadísticas:')
print(df[vars_numericas].describe().loc[['mean', 'std', 'min', 'max']])
print('\nMIN-MAX - Estadísticas:')
print(df_minmax.describe().loc[['mean', 'std', 'min', 'max']])
print('\nZ-SCORE - Estadísticas:')
print(df_zscore.describe().loc[['mean', 'std', 'min', 'max']])

ORIGINAL - Estadísticas:
         tenure  MonthlyCharges  TotalCharges
mean  32.371149       64.761692   2281.916928
std   24.559481       30.090047   2265.270398
min    0.000000       18.250000     18.800000
max   72.000000      118.750000   8684.800000

MIN-MAX - Estadísticas:
        tenure  MonthlyCharges  TotalCharges
mean  0.449599        0.462803      0.261149
std   0.341104        0.299403      0.261397
min   0.000000        0.000000      0.000000
max   1.000000        1.000000      1.000000

Z-SCORE - Estadísticas:
            tenure  MonthlyCharges  TotalCharges
mean -2.421273e-17   -6.406285e-17 -1.488074e-17
std   1.000071e+00    1.000071e+00  1.000071e+00
min  -1.318165e+00   -1.545860e+00 -9.991203e-01
max   1.613701e+00    1.794352e+00  2.826743e+00


### ✏️ Pregunta 3
**¿Cuál es la diferencia entre normalización Min-Max y estandarización Z-score? ¿Cuándo usarías cada una?**

> **Respuesta:**
  **Min-Max** escala todos los valores al rango [0, 1]. Úsala cuando el algoritmo requiere ese rango (redes neuronales, KNN) y los datos no tienen outliers extremos.
  **Z-score** transforma a media=0 y std=1. Úsala cuando hay outliers moderados o cuando el algoritmo asume distribución normal (regresión, SVM, PCA).

---
## 🔹 Paso 4: Codificación de variables categóricas
*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

In [ ]:
vars_categoricas = ['gender', 'Partner', 'Dependents', 'PhoneService',
                    'InternetService', 'Contract', 'PaymentMethod']

print('CARDINALIDAD DE VARIABLES CATEGÓRICAS:')
for var in vars_categoricas:
    print(f'  {var}: {df[var].nunique()} categorías')

df_encoded = pd.get_dummies(df[vars_categoricas], drop_first=True)
print(f'\nDataset después de One-Hot: {df_encoded.shape[1]} columnas')
print(df_encoded.columns.tolist()[:10])

CARDINALIDAD DE VARIABLES CATEGÓRICAS:
  gender: 2 categorías
  Partner: 2 categorías
  Dependents: 2 categorías
  PhoneService: 2 categorías
  InternetService: 3 categorías
  Contract: 3 categorías
  PaymentMethod: 4 categorías

Dataset después de One-Hot: 11 columnas
['gender_Male', 'Partner_Yes', 'Dependents_Yes', 'PhoneService_Yes', 'InternetService_Fiber optic', 'InternetService_No', 'Contract_One year', 'Contract_Two year', 'PaymentMethod_Credit card (automatic)', 'PaymentMethod_Electronic check']


### ✏️ Pregunta 4
**¿Por qué es importante el parámetro `drop_first=True` en One-Hot Encoding? ¿Qué problema evita?**

**Respuesta:** *Evita la multicolinealidad perfecta (trampa de la variable dummy). Si una variable tiene k categorías y se crean k columnas, la última es combinación lineal de las demás, lo que rompe modelos como regresión lineal/logística. Con drop_first=True se eliminan k−1 columnas, manteniendo la información completa sin redundancia.*

---
## 🔹 Paso 5: Feature Engineering
*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

In [ ]:
# Crear nuevas variables (Feature Engineering)
df['tenure_group'] = pd.cut(df['tenure'], bins=[0, 12, 24, 48, 72],
                           labels=['Nuevo', 'Corto', 'Medio', 'Largo'])

df['num_servicios'] = (df['PhoneService'] == 'Yes').astype(int) + \
                      (df['InternetService'] != 'No').astype(int) + \
                      (df['OnlineSecurity'] == 'Yes').astype(int) + \
                      (df['TechSupport'] == 'Yes').astype(int) + \
                      (df['StreamingTV'] == 'Yes').astype(int)

df['cargo_por_mes'] = df['TotalCharges'] / (df['tenure'] + 1)

print('NUEVAS VARIABLES CREADAS:')
print(df[['tenure_group', 'num_servicios', 'cargo_por_mes']].head(10))

NUEVAS VARIABLES CREADAS:
  tenure_group  num_servicios  cargo_por_mes
0        Nuevo              1      14.925000
1        Medio              3      53.985714
2        Nuevo              3      36.050000
3        Medio              3      40.016304
4        Nuevo              2      50.550000
5        Nuevo              3      91.166667
6        Corto              3      84.756522
7        Nuevo              2      27.445455
8        Medio              4     105.036207
9        Largo              3      55.364286


### ✏️ Pregunta 5
**¿Por qué es útil crear la variable `num_servicios`? ¿Qué información aporta al modelo de churn?**

**Respuesta:** *Consolida en una sola variable cuántos servicios contratados tiene el cliente. Un cliente con más servicios tiene mayor engagement y costo de cambio (switching cost), por lo que es menos probable que cancele. Simplifica la información de 5 variables binarias en una sola escala ordinal, facilitando la captura de esa relación por el modelo.*

---
## 🔹 Paso 6: Selección de atributos
*Ref: Müller, A. C. y Guido, S. (2016). Introduction to Machine Learning with Python. Cap. 3.*

In [ ]:
from sklearn.feature_selection import SelectKBest, mutual_info_classif

df['Churn_bin'] = (df['Churn'] == 'Yes').astype(int)
X = df_encoded.values
y = df['Churn_bin'].values

selector = SelectKBest(score_func=mutual_info_classif, k=10)
selector.fit(X, y)

scores = pd.DataFrame({'Variable': df_encoded.columns, 'Score': selector.scores_})
scores = scores.sort_values('Score', ascending=False)
print('TOP 10 VARIABLES MÁS IMPORTANTES:')
print(scores.head(10))

TOP 10 VARIABLES MÁS IMPORTANTES:
                                 Variable     Score
7                       Contract_Two year  0.061470
9          PaymentMethod_Electronic check  0.045638
4             InternetService_Fiber optic  0.042807
5                      InternetService_No  0.025623
6                       Contract_One year  0.016557
10             PaymentMethod_Mailed check  0.011893
1                             Partner_Yes  0.009977
2                          Dependents_Yes  0.008399
8   PaymentMethod_Credit card (automatic)  0.008170
3                        PhoneService_Yes  0.007476


### ✏️ Pregunta 6
**¿Cuáles son las 5 variables más importantes según SelectKBest? ¿Coinciden con los resultados del Lab 3?**

> **Respuesta:**
1. Contract_Two year (0.061)
2. PaymentMethod_Electronic check (0.046)
 3. InternetService_Fiber optic (0.043)
 4. InternetService_No (0.026) 5. Contract_One year (0.017)

El tipo de contrato y el método de pago dominan, lo cual es coherente con el Lab 3: clientes mes a mes y con cheque electrónico tienen mayor tasa de churn.

---
## 🔹 Paso 7: Reducción de dimensionalidad con PCA
*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

In [ ]:
from sklearn.decomposition import PCA

X_scaled = StandardScaler().fit_transform(X)

pca = PCA(n_components=0.95)
X_pca = pca.fit_transform(X_scaled)

print(f'Componentes originales: {X.shape[1]}')
print(f'Componentes después de PCA: {X_pca.shape[1]}')
print(f'Varianza explicada acumulada: {sum(pca.explained_variance_ratio_)*100:.2f}%')

print('\nVARIANZA EXPLICADA POR COMPONENTE:')
for i, var in enumerate(pca.explained_variance_ratio_[:5]):
    print(f'  PC{i+1}: {var*100:.2f}%')

Componentes originales: 11
Componentes después de PCA: 10
Varianza explicada acumulada: 97.48%

VARIANZA EXPLICADA POR COMPONENTE:
  PC1: 21.00%
  PC2: 14.56%
  PC3: 11.28%
  PC4: 11.10%
  PC5: 10.51%


### ✏️ Pregunta 7
**¿Cuántos componentes se necesitan para mantener el 95% de varianza? ¿Qué ventaja tiene esto para el modelo?**

**Respuesta:** *Se necesitan 10 componentes (de 11 originales) para explicar el 97.48% de varianza. La reducción mínima indica que las variables están relativamente poco correlacionadas. La ventaja principal es eliminar ruido estadístico y evitar la maldición de la dimensionalidad, aunque en este caso el beneficio de compresión es bajo.*

---
## 🔹 Paso 8: Pipeline completo de preparación
*Ref: Géron, A. (2022). Hands-On Machine Learning. Cap. 2.*

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('pca', PCA(n_components=10))
])

X_transformed = pipeline.fit_transform(X)
print(f'Dataset transformado: {X_transformed.shape}')
print('\nPipeline creado exitosamente. Listo para modelado.')

Dataset transformado: (7043, 10)

Pipeline creado exitosamente. Listo para modelado.


### ✏️ Pregunta 8
**¿Por qué es mejor usar un Pipeline que aplicar cada transformación por separado?**

**Respuesta:** *El Pipeline garantiza: (1) reproducibilidad — mismo orden siempre; (2) evitar data leakage — el scaler se ajusta solo con datos de entrenamiento; (3) simplicidad operativa — un solo objeto para fit/transform/predict; (4) compatibilidad con GridSearchCV para optimización de hiperparámetros en todo el flujo.*

---
# ACTIVIDAD 3: Caso de Estudio — Pipeline para modelo de Churn

> **Contexto:** El equipo de Data Science de TelcoPerú necesita preparar el dataset para entrenar un modelo predictivo de churn. Documenta y justifica cada decisión del pipeline.

### ✏️ Pregunta A
**Describe las 5 etapas del pipeline que aplicaste. ¿En qué orden se ejecutan y por qué?**

> **Respuesta:**
1. Imputación — Rellena nulos con mediana (debe ir primero para que no fallen pasos posteriores).
2. Codificación — One-Hot Encoding de variables categóricas (antes de escalar para que las dummies sean numéricas).
3. Escalado — StandardScaler estandariza todas las variables numéricas (necesario antes de PCA).
4. Selección / Reducción — PCA o SelectKBest reduce dimensiones (después de escalar para que los componentes sean comparables).
5. Modelado — Algoritmo clasificador entrenado con datos limpios y transformados.

### ✏️ Pregunta B
**¿Qué es Data Leakage y cómo lo evitas al dividir el dataset en train/test antes de normalizar?**

>**Respuesta:** Data Leakage ocurre cuando información del conjunto de prueba "contamina" el entrenamiento, generando métricas infladas que no se replican en producción. Se evita dividiendo el dataset en train/test antes de cualquier transformación, y ajustando todos los transformadores (scaler, imputer, PCA) únicamente con los datos de entrenamiento (fit en train, transform en ambos). El Pipeline de scikit-learn aplica esto automáticamente.

### ✏️ Pregunta C
**Compara SelectKBest con PCA. ¿En qué situaciones preferirías cada técnica?**

>**Respuesta:**

**SelectKBest** conserva variables originales e interpretables. Preferible cuando necesitas explicar qué variables impactan el churn (reportes de negocio, cumplimiento regulatorio).

**PCA** crea componentes abstractos combinando todas las variables. Preferible cuando hay alta correlación entre variables y el objetivo es solo rendimiento predictivo, sin importar la interpretabilidad.

### ✏️ Pregunta D
**¿Qué variables nuevas creaste mediante Feature Engineering? ¿Por qué son útiles para predecir churn?**

>**Respuesta:**

**tenure_group** Discretiza la antigüedad en 4 segmentos (Nuevo/Corto/Medio/Largo). Captura relaciones no lineales: los clientes nuevos tienen mayor churn.

**num_servicios** Suma de servicios contratados. Mayor número = mayor arraigo = menor churn.

**cargo_por_mes** TotalCharges / (tenure+1). Normaliza el cargo por tiempo, identificando clientes que pagan mucho en poco tiempo, señal de potencial insatisfacción.

### ✏️ Pregunta E
**Si aplicaras este pipeline a 1 millón de clientes nuevos cada mes, ¿cómo garantizarías reproducibilidad?**

>**Respuesta:**

1. **Serializar el pipeline** con joblib/pickle después del entrenamiento para que los mismos parámetros (medias, escalas, componentes) se apliquen en producción.
2. **Control de versiones** del pipeline y del dataset de entrenamiento (MLflow, DVC) para trazabilidad.
3. **Validación automática** en cada batch mensual: verificar que las distribuciones de entrada no hayan derivado (data drift) usando métricas como KL divergence o PSI.

---
# 📝 CONCLUSIONES

1. La preparación de datos es la etapa más crítica del ciclo CRISP-DM: un pipeline bien construido con imputación, codificación, escalado y reducción dimensiones garantiza modelos robustos y reproducibles.

2. El Feature Engineering aportó variables (num_servicios, cargo_por_mes, tenure_group) con mayor poder explicativo que las variables originales, confirmando que el conocimiento del negocio es tan importante como las técnicas estadísticas.

3. Evitar el Data Leakage mediante un Pipeline formal es esencial para que el rendimiento medido en validación sea representativo del comportamiento real en producción con clientes nuevos.

---
### 📚 Referencias
- Géron, A. (2022). *Hands-On Machine Learning* (3.ª ed.). O'Reilly Media.
- Gironés Roig, J. et al. (2017). *Minería de datos: modelos y algoritmos*. Editorial UOC.
- Ilyas, I. F. y Chu, X. (2019). *Cleaning Data for Data Science*. ACM.
- Müller, A. C. y Guido, S. (2016). *Introduction to Machine Learning with Python*. O'Reilly Media.

---
*Minería de Datos — Semana 4 | TECSUP 2026-I | Prof. Pilar Rocío Sayán Mejía*